# Negative Prompts Demo — Reduce False Positives with Background Masks

This notebook demonstrates how to use **negative masks** (background annotations)
to suppress false positive detections across InstantLearn models.

**Workflow:**
1. Provide a reference image with a **foreground mask** (object of interest) and a **background mask** (region to exclude)
2. The library extracts **negative point prompts** from the background mask using centroid + furthest-point sampling
3. During inference, SAM uses these negative points to avoid the excluded region

**Key Concepts:**
- `BACKGROUND_CATEGORY_ID = -1` — reserved category ID for negative/background masks
- `BACKGROUND_CATEGORY = "background"` — reserved category name
- Supported models: **Matcher**, **PerDino**, **SoftMatcher**, **SAM3**

> **Note:** Run this notebook from the `library/` directory with the virtual environment activated.

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

from instantlearn.components.negative_prompts import NegativeMaskToPoints
from instantlearn.data import Sample
from instantlearn.data.base.sample import BACKGROUND_CATEGORY, BACKGROUND_CATEGORY_ID
from instantlearn.data.utils.image import read_image, read_mask
from instantlearn.visualizer import render_predictions, setup_colors

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# Paths — adjust if running from a different directory
ASSETS_DIR = Path("assets")
DEVICE = "cuda"  # Change to "cuda" or "xpu" for GPU inference

print(f"BACKGROUND_CATEGORY = {BACKGROUND_CATEGORY!r}")
print(f"BACKGROUND_CATEGORY_ID = {BACKGROUND_CATEGORY_ID}")
print(f"Device: {DEVICE}")

## 1. Load Reference Image and Foreground Mask

We use a COCO elephant image and its corresponding mask as the reference. The foreground mask highlights the object of interest.

In [ ]:
ref_image_path = str(ASSETS_DIR / "coco" / "000000286874.jpg")
ref_mask_path = str(ASSETS_DIR / "coco" / "000000286874_mask.png")

# Load and display reference image + foreground mask
ref_image = read_image(ref_image_path, as_tensor=False)  # (H, W, C) numpy
fg_mask = read_mask(ref_mask_path, as_tensor=False)  # (H, W) numpy
h, w = ref_image.shape[:2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(ref_image)
axes[0].set_title("Reference Image")
axes[0].axis("off")

axes[1].imshow(ref_image)
axes[1].imshow(fg_mask, alpha=0.5, cmap="Greens")
axes[1].set_title("Foreground Mask (object)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"Image shape: {ref_image.shape}")
print(f"Foreground mask shape: {fg_mask.shape}, pixels: {fg_mask.sum()}")

## 2. Create a Background (Negative) Mask

A background mask marks a region the model should **exclude** from detections. This is the key input for negative prompts.

In a real application, the user would draw this mask interactively (e.g., right-click annotation). Here we create a synthetic one covering the lower-right quadrant.

In [ ]:
# Create a background mask covering the lower-right quadrant
bg_region = (h * 3 // 4, w * 3 // 4, h, w)  # (y1, x1, y2, x2)
bg_mask = np.zeros((h, w), dtype=np.uint8)
y1, x1, y2, x2 = bg_region
bg_mask[y1:y2, x1:x2] = 1

# Visualize foreground + background masks together
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(ref_image)
axes[0].set_title("Reference Image")
axes[0].axis("off")

axes[1].imshow(ref_image)
axes[1].imshow(fg_mask, alpha=0.5, cmap="Greens")
axes[1].set_title("Foreground Mask (green)")
axes[1].axis("off")

axes[2].imshow(ref_image)
axes[2].imshow(fg_mask, alpha=0.4, cmap="Greens")
axes[2].imshow(bg_mask, alpha=0.4, cmap="Reds")
axes[2].set_title("Foreground (green) + Background (red)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print(f"Background mask region: y=[{y1}:{y2}], x=[{x1}:{x2}]")
print(f"Background mask pixels: {bg_mask.sum()}")

## 3. Understand NegativeMaskToPoints

The `NegativeMaskToPoints` component converts background masks into point prompts that SAM understands. It uses **centroid + furthest-point sampling** for good spatial coverage:

1. **Centroid** of the mask is the first point
2. Remaining points are selected via **furthest-point sampling** — each new point maximizes distance from all previously selected points

All points get `label=0` (SAM background convention).

In [ ]:
NUM_NEGATIVE_POINTS = 5

# Convert background mask to negative point prompts
converter = NegativeMaskToPoints(num_points_per_mask=NUM_NEGATIVE_POINTS)
neg_points, neg_labels = converter(torch.as_tensor(bg_mask, dtype=torch.bool).unsqueeze(0))

print(f"Sampled {neg_points.shape[0]} negative points from background mask")
print(f"Points shape: {neg_points.shape}  (M, 2) in (x, y) coords")
print(f"Labels shape: {neg_labels.shape}  all zeros = background for SAM")
print("\nSampled points (x, y):")
for i, pt in enumerate(neg_points):
    print(f"  Point {i}: ({pt[0]:.1f}, {pt[1]:.1f})  label={neg_labels[i]:.0f}")

# Visualize sampled points on the background mask
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.imshow(ref_image)
ax.imshow(bg_mask, alpha=0.3, cmap="Reds")

for i, pt in enumerate(neg_points):
    x, y = pt[0].item(), pt[1].item()
    marker = "o" if i == 0 else "x"  # centroid = circle, others = cross
    label = "centroid" if i == 0 else ("FPS points" if i == 1 else None)
    ax.plot(x, y, marker, color="red", markersize=12, markeredgewidth=2, label=label)

ax.legend(fontsize=10)
ax.set_title(f"Sampled Negative Points ({NUM_NEGATIVE_POINTS}) on Background Mask")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Build the Reference Sample

To use negative prompts, create a `Sample` with **both** foreground and background masks. The key is setting:
- Foreground masks: `category_ids >= 0` (any non-negative integer)
- Background masks: `category_ids = -1` (`BACKGROUND_CATEGORY_ID`)

The library automatically separates them: foreground masks go to feature extraction, background masks go to negative point sampling.

In [ ]:
# Reference sample with BOTH foreground and background masks
ref_sample_with_neg = Sample(
    image_path=ref_image_path,
    masks=np.stack([fg_mask, bg_mask]),  # Stack both masks: (2, H, W)
    categories=["object", BACKGROUND_CATEGORY],  # "object" + "background"
    category_ids=np.array([1, BACKGROUND_CATEGORY_ID]),  # 1 = foreground, -1 = background
    is_reference=[True, True],
)

# For comparison: baseline sample with foreground only
ref_sample_baseline = Sample(
    image_path=ref_image_path,
    mask_paths=ref_mask_path,
    categories=["object"],
    category_ids=np.array([1]),
)

print("Sample with negative prompts:")
print(f"  masks shape: {ref_sample_with_neg.masks.shape}")
print(f"  categories: {ref_sample_with_neg.categories}")
print(f"  category_ids: {ref_sample_with_neg.category_ids}")
print(f"  has_background: {ref_sample_with_neg.has_background()}")

print("\nBaseline sample (foreground only):")
print(f"  categories: {ref_sample_baseline.categories}")
print(f"  category_ids: {ref_sample_baseline.category_ids}")
print(f"  has_background: {ref_sample_baseline.has_background()}")

## 5. Run Matcher: Baseline vs. Negative Prompts

We run Matcher **twice** — once without and once with negative prompts — and compare the predictions.

In [ ]:
from instantlearn.models import Matcher

target_paths = [
    str(ASSETS_DIR / "coco" / "000000390341.jpg"),
    str(ASSETS_DIR / "coco" / "000000173279.jpg"),
    str(ASSETS_DIR / "coco" / "000000267704.jpg"),
]

# --- Baseline (no negative prompts) ---
model_baseline = Matcher(device=DEVICE, num_negative_points=NUM_NEGATIVE_POINTS)
model_baseline.fit(ref_sample_baseline)

print("Baseline negative embedding:", model_baseline._negative_embedding)
preds_baseline = model_baseline.predict(target_paths)

# --- With negative prompts ---
model_with_neg = Matcher(device=DEVICE, num_negative_points=NUM_NEGATIVE_POINTS)
model_with_neg.fit(ref_sample_with_neg)

neg_emb = model_with_neg._negative_embedding
print(f"Negative embedding extracted: {neg_emb.shape if neg_emb is not None else None}")
preds_with_neg = model_with_neg.predict(target_paths)

# Summary
print("\n--- Results Summary ---")
for i, target in enumerate(target_paths):
    name = Path(target).stem
    n_base = preds_baseline[i]["pred_masks"].shape[0]
    n_neg = preds_with_neg[i]["pred_masks"].shape[0]
    print(f"  {name}: baseline={n_base} masks, with_neg={n_neg} masks (delta={n_neg - n_base:+d})")

## 6. Visualize: Baseline vs. Negative Prompts

Compare prediction masks side-by-side for each target image.

In [ ]:
color_map = setup_colors({1: "object"})

for i, target_path in enumerate(target_paths):
    img_bgr = cv2.imread(target_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    vis_baseline = render_predictions(img_rgb, preds_baseline[i], color_map)
    vis_with_neg = render_predictions(img_rgb, preds_with_neg[i], color_map)

    n_base = preds_baseline[i]["pred_masks"].shape[0]
    n_neg = preds_with_neg[i]["pred_masks"].shape[0]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_rgb)
    axes[0].set_title(f"Original — {Path(target_path).stem}")
    axes[0].axis("off")

    axes[1].imshow(vis_baseline)
    axes[1].set_title(f"Baseline: {n_base} masks")
    axes[1].axis("off")

    axes[2].imshow(vis_with_neg)
    axes[2].set_title(f"With Negative Prompts: {n_neg} masks")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

## 7. Multi-Model: Same Interface Across Models

The negative prompt API is consistent across point-based models. The same `Sample` with `BACKGROUND_CATEGORY_ID` works for Matcher, PerDino, and SoftMatcher.

In [ ]:
from instantlearn.models.per_dino import PerDino

# Same sample works for all point-based models
models = {
    "Matcher": Matcher(device=DEVICE, num_negative_points=NUM_NEGATIVE_POINTS),
    "PerDino": PerDino(device=DEVICE, num_negative_points=NUM_NEGATIVE_POINTS),
}

target_path = target_paths[0]

print("--- Multi-Model Negative Prompts ---\n")
for name, model in models.items():
    # Same fit call for all models
    model.fit(ref_sample_with_neg)

    neg_emb = model._negative_embedding
    has_neg = neg_emb is not None
    print(f"{name}:")
    print(f"  Negative embedding extracted: {neg_emb.shape if has_neg else None}")

    # Same predict call
    preds = model.predict(target_path)
    n_masks = preds[0]["pred_masks"].shape[0]
    print(f"  Predicted masks: {n_masks}\n")

## 8. Note on `num_negative_points`

For **Matcher** and **PerDino**, the negative prompt mechanism works in feature space—the entire background mask region is pooled into a single negative embedding and subtracted from similarity maps. The `num_negative_points` parameter does **not** affect their behavior.

For **SAM3**, `num_negative_points` controls how many coordinate-based negative points are sampled from the background mask and fed to the geometry encoder. More points = stronger spatial suppression.

In [ ]:
# Verify that num_negative_points does NOT affect Matcher (feature-based approach)
target_path = target_paths[0]
point_counts = [1, 3, 5, 10, 15]

print("--- num_negative_points sweep (Matcher, feature-based) ---\n")
for n_pts in point_counts:
    model = Matcher(device=DEVICE, num_negative_points=n_pts)
    model.fit(ref_sample_with_neg)
    preds = model.predict(target_path)
    n_masks = preds[0]["pred_masks"].shape[0]
    print(f"  num_negative_points={n_pts:2d} → {n_masks} masks")

print("\n↑ All values should be the same — Matcher uses feature-space approach, not point sampling.")

## 9. Using `split_foreground_background()`

The `Sample` class provides a utility to split annotations into foreground and background sub-samples. This is useful when you need to inspect or process them separately.

In [ ]:
fg_sample, bg_sample = ref_sample_with_neg.split_foreground_background()

print("Foreground sample:")
print(f"  categories: {fg_sample.categories}")
print(f"  category_ids: {fg_sample.category_ids}")
print(f"  masks shape: {fg_sample.masks.shape}")

print("\nBackground sample:")
print(f"  categories: {bg_sample.categories}")
print(f"  category_ids: {bg_sample.category_ids}")
print(f"  masks shape: {bg_sample.masks.shape}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(ref_image)
axes[0].imshow(
    fg_sample.masks[0] if isinstance(fg_sample.masks, np.ndarray) else fg_sample.masks[0].numpy(),
    alpha=0.5,
    cmap="Greens",
)
axes[0].set_title(f"Foreground: {fg_sample.categories}")
axes[0].axis("off")

axes[1].imshow(ref_image)
axes[1].imshow(
    bg_sample.masks[0] if isinstance(bg_sample.masks, np.ndarray) else bg_sample.masks[0].numpy(),
    alpha=0.5,
    cmap="Reds",
)
axes[1].set_title(f"Background: {bg_sample.categories}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 10. Multiple Background Masks

You can provide multiple background masks (e.g., multiple distractors). For Matcher/PerDino, all background mask regions are pooled together into a single negative embedding.

In [ ]:
# Two separate background regions
bg_mask_1 = np.zeros((h, w), dtype=np.uint8)
bg_mask_1[0 : h // 4, 0 : w // 4] = 1  # Top-left corner

bg_mask_2 = np.zeros((h, w), dtype=np.uint8)
bg_mask_2[h * 3 // 4 : h, w * 3 // 4 : w] = 1  # Bottom-right corner

# Sample with 1 foreground + 2 background masks
ref_multi_bg = Sample(
    image_path=ref_image_path,
    masks=np.stack([fg_mask, bg_mask_1, bg_mask_2]),
    categories=["object", BACKGROUND_CATEGORY, BACKGROUND_CATEGORY],
    category_ids=np.array([1, BACKGROUND_CATEGORY_ID, BACKGROUND_CATEGORY_ID]),
    is_reference=[True, True, True],
)

model_multi_bg = Matcher(device=DEVICE, num_negative_points=3)
model_multi_bg.fit(ref_multi_bg)

neg_emb = model_multi_bg._negative_embedding
has_neg = neg_emb is not None
print(f"2 background masks → negative embedding: {neg_emb.shape if has_neg else None}")

# Visualize the two background regions
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.imshow(ref_image)
ax.imshow(fg_mask, alpha=0.4, cmap="Greens")
ax.imshow(bg_mask_1, alpha=0.4, cmap="Reds")
ax.imshow(bg_mask_2, alpha=0.4, cmap="Oranges")

ax.set_title("Multiple Background Masks (pooled into single negative embedding)")
ax.axis("off")
plt.tight_layout()
plt.show()

## Summary

| Concept | Details |
|---|---|
| **Background category ID** | `BACKGROUND_CATEGORY_ID = -1` |
| **Background category name** | `BACKGROUND_CATEGORY = "background"` |
| **How to mark a mask as negative** | Set its `category_id = -1` in the Sample |
| **How it works (Matcher/PerDino)** | Background mask features are averaged into a negative embedding, then subtracted from similarity maps during prediction |
| **How it works (SAM3)** | Centroid + furthest-point sampling → coordinate-based negative points fed to geometry encoder |
| **Supported models** | Matcher, PerDino, SoftMatcher, SAM3 |
| **Not supported** | GroundedSAM (text-only, no point-based prompts) |
| **Key parameter** | `num_negative_points` (default: 5) — used by SAM3; Matcher/PerDino use feature-space approach |